# 06 - Camada Gold: Modelagem Dimensional, KPIs e Destino Duplo (Modo Append)
**Squad 2 — Real Time for Business | Unificação Dupla 1 & Dupla 5**  
**Integrantes:** Zaiden Emiliano Segundo Seleme & Lucas Sousa Santos Oliveira  
**Data Marts de Escopo:**
1. `gold_dim_produtos` (Grupo 1)
2. `gold_metricas_categorias` (Grupo 1)
3. `gold_logistica_pedidos_rota` (Grupo 5)
4. `gold_performance_transportadoras` (Grupo 5)
5. `gold_distribuicao_geografica_clientes` (Grupo 5)

**Branch:** `feat/squad2-dupla1e5`

---

### Diretrizes Arquiteturais da Camada Gold:
1. **Consumo Exclusivo Silver-to-Gold:** Leitura restrita aos dados curados da Camada Silver em Delta Lake. Zero leitura de raw/bronze.
2. **Modelagem Dimensional e KPIs:**
   * **Dimensão de Produtos:** Desnormalização com hierarquia pai/filha de categorias, segmentação por faixas de preço e flags de disponibilidade.
   * **Métricas de Categorias:** Agregações de catálogo, taxa de disponibilidade, ticket médio e dispersão.
   * **Logística de Entregas:** Throughput diário de saídas para rota, entregas concluídas, falhas operacionais e taxa de sucesso percentual.
   * **Scorecard de Transportadoras:** Ranking de eficiência operacional por data de evento via `dense_rank()`.
   * **Inteligência Geográfica:** Distribuição e densidade de clientes e endereços cadastrados por estado (UF).
3. **Destino Duplo Mandatório (*Dual Sink*) em Modo APPEND:**
   * Conforme alinhamento técnico oficial com o Tech Lead na reunião, a persistência da Gold opera em **modo `append`**.
   * Permite a **historização contínua de snapshots analíticos e métricas** carimbadas com `gold_processed_at` para análise de tendência temporal no Power BI / BI.
   * **ADLS Gen2:** Tabelas analíticas em Delta Lake com modo `append`, ACID e compactação automática (`autoCompact`, `optimizeWrite`).
   * **Azure SQL Server:** Replicação relacional acumulada no schema `squad2` (modo `append`) para alimentar os dashboards executivos.
4. **Auditoria de Fechamento Matemático:** Validação estrita de integridade para apresentação ao PO.

In [ ]:
# Instalação de dependências no escopo da sessão Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity pandas pyodbc --quiet

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType, TimestampType
)

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env")
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

# Credenciais ADLS Gen2
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

# Credenciais SQL Server
sql_host = os.getenv("SQL_HOST", "srv-database-intership.database.windows.net")
sql_database = os.getenv("SQL_DATABASE", "internshipDatabase")
sql_user = os.getenv("SQL_USERNAME", "estagiario_user")
sql_password = os.getenv("SQL_PASSWORD")

assert client_id and client_secret and sql_password, "Credenciais ADLS/SQL incompletas no .env"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

jdbc_url = f"jdbc:sqlserver://{sql_host}:1433;database={sql_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net"

caminhos = {
    "silver_produtos": f"{base_squad}/silver/ecommerce_produtos",
    "silver_categorias": f"{base_squad}/silver/ecommerce_categorias",
    "silver_rastreamento": f"{base_squad}/silver/ecommerce_rastreamento_entregas",
    "silver_enderecos": f"{base_squad}/silver/ecommerce_enderecos",

    "gold_dim_produtos": f"{base_squad}/gold/gold_dim_produtos",
    "gold_metricas_categorias": f"{base_squad}/gold/gold_metricas_categorias",
    "gold_pedidos_rota": f"{base_squad}/gold/gold_logistica_pedidos_rota",
    "gold_performance_transportadoras": f"{base_squad}/gold/gold_performance_transportadoras",
    "gold_distribuicao_geografica": f"{base_squad}/gold/gold_distribuicao_geografica_clientes"
}

def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

# Schemas de contingência para leitura defensiva
schema_spark_produtos = StructType([
    StructField("sku", StringType(), True),
    StructField("nome_produto", StringType(), True),
    StructField("descricao", StringType(), True),
    StructField("id_categoria", LongType(), True),
    StructField("preco_lista", DoubleType(), True),
    StructField("unidade_medida", StringType(), True),
    StructField("nome_marca", StringType(), True),
    StructField("is_ativo", BooleanType(), True)
])

schema_spark_categorias = StructType([
    StructField("id_categoria", LongType(), True),
    StructField("nome_categoria", StringType(), True),
    StructField("id_categoria_pai", LongType(), True),
    StructField("nome_categoria_pai", StringType(), True),
    StructField("tipo_categoria", StringType(), True)
])

schema_spark_rastreamento = StructType([
    StructField("id_rastreamento", LongType(), True),
    StructField("id_pedido_ecommerce", LongType(), True),
    StructField("codigo_rastreio", StringType(), True),
    StructField("id_transportadora", LongType(), True),
    StructField("nome_transportadora", StringType(), True),
    StructField("status_entrega", StringType(), True),
    StructField("dt_evento", TimestampType(), True),
    StructField("observacao", StringType(), True)
])

schema_spark_enderecos = StructType([
    StructField("id_endereco", LongType(), True),
    StructField("id_cliente", LongType(), True),
    StructField("logradouro", StringType(), True),
    StructField("numero", LongType(), True),
    StructField("complemento", StringType(), True),
    StructField("bairro", StringType(), True),
    StructField("cep", StringType(), True),
    StructField("cidade", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("apelido", StringType(), True),
    StructField("is_principal", BooleanType(), True)
])

print("Ambiente e caminhos da Camada Gold preparados com proteção de leitura!")

In [ ]:
def carregar_silver_seguro(caminho, schema_spark, nome):
    if not tabela_delta_existe(caminho):
        print(f"  [AVISO] Tabela Silver '{nome}' não encontrada. Retornando DataFrame vazio para continuidade da esteira.")
        return spark.createDataFrame([], schema_spark)
    return spark.read.format("delta").options(**adls_options).load(caminho)

print("Lendo tabelas curadas da Camada Silver de forma defensiva...")
df_silver_produtos = carregar_silver_seguro(caminhos["silver_produtos"], schema_spark_produtos, "ecommerce_produtos")
df_silver_categorias = carregar_silver_seguro(caminhos["silver_categorias"], schema_spark_categorias, "ecommerce_categorias")
df_silver_rastreamento = carregar_silver_seguro(caminhos["silver_rastreamento"], schema_spark_rastreamento, "ecommerce_rastreamento_entregas")
df_silver_enderecos = carregar_silver_seguro(caminhos["silver_enderecos"], schema_spark_enderecos, "ecommerce_enderecos")

print(f"  Produtos:     {df_silver_produtos.count()} registros carregados.")
print(f"  Categorias:   {df_silver_categorias.count()} registros carregados.")
print(f"  Rastreamento: {df_silver_rastreamento.count()} registros carregados.")
print(f"  Endereços:    {df_silver_enderecos.count()} registros carregados.")

## 1. Leitura das Tabelas Curadas da Camada Silver (Delta Lake)

In [ ]:
print("Lendo tabelas curadas da Camada Silver...")
df_silver_produtos = spark.read.format("delta").options(**adls_options).load(caminhos["silver_produtos"])
df_silver_categorias = spark.read.format("delta").options(**adls_options).load(caminhos["silver_categorias"])
df_silver_rastreamento = spark.read.format("delta").options(**adls_options).load(caminhos["silver_rastreamento"])
df_silver_enderecos = spark.read.format("delta").options(**adls_options).load(caminhos["silver_enderecos"])

print(f"  Produtos:     {df_silver_produtos.count()} registros carregados.")
print(f"  Categorias:   {df_silver_categorias.count()} registros carregados.")
print(f"  Rastreamento: {df_silver_rastreamento.count()} registros carregados.")
print(f"  Endereços:    {df_silver_enderecos.count()} registros carregados.")

## 2. Construção dos Data Marts de Produtos e Categorias (Grupo 1)

In [ ]:
# 1. Data Mart 1: gold_dim_produtos
df_cat_lookup = df_silver_categorias.select(
    F.col("id_categoria"),
    F.col("nome_categoria"),
    F.col("id_categoria_pai"),
    F.col("nome_categoria_pai"),
    F.col("tipo_categoria")
)

df_gold_dim_produtos = (
    df_silver_produtos
    .join(F.broadcast(df_cat_lookup), on="id_categoria", how="left")
    .withColumn("status_disponibilidade", F.when(F.col("is_ativo") == True, F.lit("Ativo")).otherwise(F.lit("Inativo")))
    .withColumn(
        "faixa_preco",
        F.when(F.col("preco_lista") < 50, F.lit("1. Economica (< R$ 50)"))
        .when((F.col("preco_lista") >= 50) & (F.col("preco_lista") < 200), F.lit("2. Padrao (R$ 50 - 200)"))
        .when((F.col("preco_lista") >= 200) & (F.col("preco_lista") < 1000), F.lit("3. Premium (R$ 200 - 1000)"))
        .otherwise(F.lit("4. Luxo (>= R$ 1000)"))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

print("Data Mart 1 (gold_dim_produtos) gerado:")
display(df_gold_dim_produtos.limit(5))

# 2. Data Mart 2: gold_metricas_categorias
df_gold_metricas_categorias = (
    df_gold_dim_produtos
    .groupBy(
        "id_categoria",
        "nome_categoria",
        "id_categoria_pai",
        "nome_categoria_pai",
        "tipo_categoria"
    )
    .agg(
        F.count("sku").alias("total_skus_cadastrados"),
        F.sum(F.when(F.col("is_ativo") == True, 1).otherwise(0)).alias("total_skus_ativos"),
        F.sum(F.when(F.col("is_ativo") == False, 1).otherwise(0)).alias("total_skus_inativos"),
        F.round(F.avg("preco_lista"), 2).alias("preco_medio_lista"),
        F.round(F.min("preco_lista"), 2).alias("preco_minimo_lista"),
        F.round(F.max("preco_lista"), 2).alias("preco_maximo_lista"),
        F.round(F.stddev("preco_lista"), 2).alias("desvio_padrao_preco")
    )
    .withColumn(
        "taxa_disponibilidade_pct",
        F.round((F.col("total_skus_ativos") / F.col("total_skus_cadastrados")) * 100, 2)
    )
    .withColumn(
        "is_categoria_raiz",
        F.when(F.col("id_categoria_pai").isNull(), F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy(F.desc("total_skus_cadastrados"))
)

print("Data Mart 2 (gold_metricas_categorias) gerado:")
display(df_gold_metricas_categorias.limit(5))

## 3. Construção dos Data Marts de Logística e Endereços (Grupo 5)

In [ ]:
# 3. Data Mart 3: gold_logistica_pedidos_rota
df_gold_pedidos_rota = (
    df_silver_rastreamento
    .withColumn("data_evento", F.to_date("dt_evento"))
    .groupBy("data_evento")
    .agg(
        F.count("id_rastreamento").alias("total_eventos"),
        F.countDistinct("id_pedido_ecommerce").alias("total_pedidos_unicos"),
        F.sum(F.when(F.col("status_entrega") == "coletado", 1).otherwise(0)).alias("total_coletados"),
        F.sum(F.when(F.col("status_entrega") == "em_transito", 1).otherwise(0)).alias("total_em_transito"),
        F.sum(F.when(F.col("status_entrega") == "saiu_para_entrega", 1).otherwise(0)).alias("total_em_rota"),
        F.sum(F.when(F.col("status_entrega") == "entregue", 1).otherwise(0)).alias("total_entregues"),
        F.sum(F.when(F.col("status_entrega").isin("falha_na_entrega", "devolvido"), 1).otherwise(0)).alias("total_falhas")
    )
    .withColumn(
        "taxa_sucesso_pct",
        F.when(
            (F.col("total_entregues") + F.col("total_falhas")) > 0,
            F.round((F.col("total_entregues") / (F.col("total_entregues") + F.col("total_falhas"))) * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy(F.desc("data_evento"))
)

print("Data Mart 3 (gold_logistica_pedidos_rota) gerado:")
display(df_gold_pedidos_rota.limit(5))

# 4. Data Mart 4: gold_performance_transportadoras
w_rank_transportadora = Window.partitionBy("data_evento").orderBy(F.desc("volume_eventos"))

df_gold_transportadoras = (
    df_silver_rastreamento
    .withColumn("data_evento", F.to_date("dt_evento"))
    .groupBy("data_evento", "id_transportadora", "nome_transportadora")
    .agg(
        F.count("id_rastreamento").alias("volume_eventos"),
        F.sum(F.when(F.col("status_entrega") == "entregue", 1).otherwise(0)).alias("total_entregues"),
        F.sum(F.when(F.col("status_entrega").isin("falha_na_entrega", "devolvido"), 1).otherwise(0)).alias("total_falhas")
    )
    .withColumn(
        "taxa_sucesso_pct",
        F.when(
            (F.col("total_entregues") + F.col("total_falhas")) > 0,
            F.round((F.col("total_entregues") / (F.col("total_entregues") + F.col("total_falhas"))) * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("rank_volume_diario", F.dense_rank().over(w_rank_transportadora))
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy("data_evento", "rank_volume_diario")
)

print("Data Mart 4 (gold_performance_transportadoras) gerado:")
display(df_gold_transportadoras.limit(5))

# 5. Data Mart 5: gold_distribuicao_geografica_clientes
total_clientes_base = df_silver_enderecos.select("id_cliente").distinct().count()

df_gold_geografia = (
    df_silver_enderecos
    .groupBy("estado")
    .agg(
        F.count("id_endereco").alias("total_enderecos_cadastrados"),
        F.countDistinct("id_cliente").alias("total_clientes_unicos"),
        F.sum(F.when(F.col("is_principal") == True, 1).otherwise(0)).alias("total_enderecos_principais")
    )
    .withColumn(
        "penetracao_clientes_pct",
        F.round((F.col("total_clientes_unicos") / F.lit(total_clientes_base)) * 100, 2)
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy(F.desc("total_clientes_unicos"))
)

print("Data Mart 5 (gold_distribuicao_geografica_clientes) gerado:")
display(df_gold_geografia.limit(5))

## 4. Persistência 1: Gravação em Delta Lake no ADLS Gen2

In [ ]:
data_marts_lake = {
    caminhos["gold_dim_produtos"]: df_gold_dim_produtos,
    caminhos["gold_metricas_categorias"]: df_gold_metricas_categorias,
    caminhos["gold_pedidos_rota"]: df_gold_pedidos_rota,
    caminhos["gold_performance_transportadoras"]: df_gold_transportadoras,
    caminhos["gold_distribuicao_geografica"]: df_gold_geografia
}

print("Persistindo Data Marts em Delta Lake no modo APPEND (ADLS Gen2)...")
for path, df_mart in data_marts_lake.items():
    (df_mart.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .save(path))
    print(f"  [OK] Gravado em modo append com sucesso em: {path}")

## 5. Persistência 2: Replicação Resiliente no Azure SQL Server (`squad2`)

In [ ]:
import pyodbc

def salvar_gold_sql(df, nome_tabela):
    if df is None or df.count() == 0:
        print(f"\n[AVISO DE PROTEÇÃO] Data Mart '{nome_tabela}' sem registros para replicar. Persistência ignorada com segurança.")
        return

    print(f"\nReplicando {nome_tabela} no Azure SQL Server (modo APPEND - {df.count()} linhas)...")
    
    # 1. Tentativa conector nativo sqlserver
    try:
        df.write \
            .format("sqlserver") \
            .option("host", sql_host) \
            .option("port", "1433") \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .option("database", sql_database) \
            .option("dbtable", nome_tabela) \
            .mode("append") \
            .save()
        print(f"  [SUCESSO] {nome_tabela} gravado em modo append via conector sqlserver.")
        return
    except Exception as e_sql:
        print(f"  Aviso (sqlserver): {e_sql}")

    # 2. Tentativa conector sqlserver via JDBC URL
    try:
        df.write \
            .format("sqlserver") \
            .option("url", jdbc_url) \
            .option("dbtable", nome_tabela) \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .mode("append") \
            .save()
        print(f"  [SUCESSO] {nome_tabela} gravado em modo append via JDBC URL.")
        return
    except Exception as e_url:
        print(f"  Aviso (url): {e_url}")

    # 3. Fallback inteligente pyodbc (modo append acumulativo)
    print("  -> Ativando fallback pyodbc fast_executemany (modo append)...")
    try:
        pdf = df.toPandas()
        conn_str = (
            f"DRIVER={{ODBC Driver 18 for SQL Server}};"
            f"SERVER={sql_host},1433;"
            f"DATABASE={sql_database};"
            f"UID={sql_user};"
            f"PWD={sql_password};"
            f"Encrypt=yes;"
            f"TrustServerCertificate=no;"
            f"Connection Timeout=30;"
        )
        with pyodbc.connect(conn_str) as conn:
            cursor = conn.cursor()
            cursor.fast_executemany = True
            
            cols = [f"[{c}]" for c in pdf.columns]
            placeholders = ", ".join(["?"] * len(pdf.columns))
            sql = f"INSERT INTO {nome_tabela} ({', '.join(cols)}) VALUES ({placeholders})"
            
            pdf_clean = pdf.where(pd.notnull(pdf), None)
            valores = [tuple(x) for x in pdf_clean.to_numpy()]
            cursor.executemany(sql, valores)
            conn.commit()
            print(f"  [SUCESSO] {len(pdf)} registros inseridos (modo append) via pyodbc em {nome_tabela}.")
    except Exception as e_py:
        raise RuntimeError(f"Falha ao persistir {nome_tabela} no SQL Server: {e_py}")

salvar_gold_sql(df_gold_dim_produtos, "squad2.gold_dim_produtos")
salvar_gold_sql(df_gold_metricas_categorias, "squad2.gold_metricas_categorias")
salvar_gold_sql(df_gold_pedidos_rota, "squad2.gold_logistica_pedidos_rota")
salvar_gold_sql(df_gold_transportadoras, "squad2.gold_performance_transportadoras")
salvar_gold_sql(df_gold_geografia, "squad2.gold_distribuicao_geografica_clientes")

## 6. Auditoria de Fechamento Contábil e Relatório Executivo Gold

In [ ]:
print("=" * 80)
print("RELATÓRIO EXECUTIVO DE FECHAMENTO ANALÍTICO - CAMADA GOLD (SQUAD 2)")
print("=" * 80)

# Auditoria Fechamento Contábil Grupo 1
chk_produtos = df_gold_metricas_categorias.select(
    F.sum("total_skus_cadastrados").alias("cadastrados"),
    F.sum("total_skus_ativos").alias("ativos"),
    F.sum("total_skus_inativos").alias("inativos")
).first()

fechamento_ok = chk_produtos["cadastrados"] == (chk_produtos["ativos"] + chk_produtos["inativos"])

print(f"Fechamento Contábil de Catálogo (Grupo 1):")
print(f"  Total SKUs Cadastrados: {chk_produtos['cadastrados']}")
print(f"  Ativos + Inativos:      {chk_produtos['ativos'] + chk_produtos['inativos']}")
print(f"  Status Fechamento:      {'[100% EXATO - FECHAMENTO CONTÁBIL CONFIRMADO]' if fechamento_ok else '[ERRO DE CONCILIAÇÃO]'}")

print(f"\nResumo dos 5 Data Marts Analíticos da Squad 2:")
print(f"  1. gold_dim_produtos:                      {df_gold_dim_produtos.count()} produtos modelados")
print(f"  2. gold_metricas_categorias:               {df_gold_metricas_categorias.count()} categorias com métricas agregadas")
print(f"  3. gold_logistica_pedidos_rota:            {df_gold_pedidos_rota.count()} dias operacionais analisados")
print(f"  4. gold_performance_transportadoras:       {df_gold_transportadoras.count()} linhas de ranking de parceiros")
print(f"  5. gold_distribuicao_geografica_clientes:  {df_gold_geografia.count()} UFs com análise de penetração")
print("=" * 80)
print("Pipeline da Camada Gold concluído e certificado com Destino Duplo!")